# chunking.py — Interactive Test

Testa `chunk_pages` com textos sintéticos e com os PDFs reais da pasta `pdf_documents`.

Certifique-se de ter instalado os requisitos:
```bash
pip install -r requirements.txt
```

In [1]:
import sys
from pathlib import Path

ROOT = Path(".").resolve().parent
PDF_DIR = ROOT / "pdf_documents"
sys.path.insert(0, str(ROOT))

print("Project root:", ROOT)
print("PDF directory:", PDF_DIR)

Project root: C:\Users\Leal\learning\Tractian_case
PDF directory: C:\Users\Leal\learning\Tractian_case\pdf_documents


In [2]:
from core.extraction import extract_pdf, PageContent
from core.chunking import chunk_pages, Chunk, _approx_tokens, CHUNK_SIZE, CHUNK_OVERLAP, MIN_CHUNK_TOKENS

print("Import OK")
print(f"Defaults — CHUNK_SIZE={CHUNK_SIZE} tokens | CHUNK_OVERLAP={CHUNK_OVERLAP} tokens | MIN_CHUNK_TOKENS={MIN_CHUNK_TOKENS} tokens")

Import OK
Defaults — CHUNK_SIZE=512 tokens | CHUNK_OVERLAP=50 tokens | MIN_CHUNK_TOKENS=50 tokens


## Helper

In [ ]:
def print_chunk_summary(chunks: list[Chunk], label: str = "") -> None:
    if label:
        print(f"=== {label} ===")
    print(f"Total chunks : {len(chunks)}")
    token_counts = [_approx_tokens(c.text) for c in chunks]
    print(f"Token sizes  : min={min(token_counts)} | max={max(token_counts)} | avg={sum(token_counts)//len(token_counts)}")
    print()

def print_chunks(chunks: list[Chunk], n: int = 5) -> None:
    for c in chunks[:n]: 
        tokens = _approx_tokens(c.text)
        print(f"[chunk {c.chunk_index}] page={c.page_number} | ~{tokens} tokens")
        print(repr(c.text[:200]))
        print()

## 1 — Texto simples: respeito a parágrafos

Verifica que o chunker usa `\n\n` como primeira separação e não quebra parágrafos desnecessariamente.

In [18]:
paragraphs = [
    "Este é o primeiro parágrafo com informações sobre motores elétricos WEG.",
    "O segundo parágrafo descreve as especificações técnicas do equipamento em detalhes.",
    "O terceiro parágrafo aborda os procedimentos de instalação e manutenção recomendados pelo fabricante.",
    "Conclusão: o motor deve ser operado dentro dos limites de temperatura indicados no manual.",
]

text = "\n\n".join(paragraphs)

pages = [PageContent(page_number=1, text=text, filename="synthetic.pdf")]
chunks = chunk_pages(pages)

print_chunk_summary(chunks, "Parágrafo simples")
print_chunks(chunks)

=== Parágrafo simples ===
Total chunks : 1
Token sizes  : min=88 | max=88 | avg=88

[chunk 0] page=1 | ~88 tokens
'Este é o primeiro parágrafo com informações sobre motores elétricos WEG.\n\nO segundo parágrafo descreve as especificações técnicas do equipamento em detalhes.\n\nO terceiro parágrafo aborda os procedimen'



In [19]:
paragraphs = [
    "Este é o primeiro parágrafo com informações sobre motores elétricos WEG.",
    "O segundo parágrafo descreve as especificações técnicas do equipamento em detalhes.",
    "O terceiro parágrafo aborda os procedimentos de instalação e manutenção recomendados pelo fabricante.",
    "Conclusão: o motor deve ser operado dentro dos limites de temperatura indicados no manual.",
]

text = "\n\n".join(paragraphs)

pages = [PageContent(page_number=1, text=text, filename="synthetic.pdf")]
chunks = chunk_pages(pages)

print_chunk_summary(chunks, "Parágrafo simples")
print_chunks(chunks)

# Todos os parágrafos são pequenos — devem ser mesclados num único chunk
assert len(chunks) >= 1, "Deve gerar pelo menos 1 chunk"
for c in chunks:
    assert _approx_tokens(c.text) <= CHUNK_SIZE, f"Chunk {c.chunk_index} excede CHUNK_SIZE"
print("OK — nenhum chunk excede CHUNK_SIZE")

=== Parágrafo simples ===
Total chunks : 1
Token sizes  : min=88 | max=88 | avg=88

[chunk 0] page=1 | ~88 tokens
'Este é o primeiro parágrafo com informações sobre motores elétricos WEG.\n\nO segundo parágrafo descreve as especificações técnicas do equipamento em detalhes.\n\nO terceiro parágrafo aborda os procedimen'

OK — nenhum chunk excede CHUNK_SIZE


## 2 — Parágrafo longo: fallback para quebra por sentença

Um único parágrafo gigante (> CHUNK_SIZE tokens) deve ser quebrado usando `. ` como separador.

In [33]:
# ~600 tokens (cada sentença ~30 tokens, 20 sentenças)
sentences = [f"Esta é a sentença número {i} do parágrafo muito longo que precisa ser dividido pelo chunker" for i in range(1, 60)]
long_paragraph = ". ".join(sentences) + "."

print(f"Tokens do parágrafo: ~{_approx_tokens(long_paragraph)}")

pages = [PageContent(page_number=1, text=long_paragraph, filename="long_para.pdf")]
chunks = chunk_pages(pages)

print_chunk_summary(chunks, "Parágrafo longo")
print_chunks(chunks)

# assert <condição>, "mensagem de erro"
assert _approx_tokens(long_paragraph) > CHUNK_SIZE , "Parágrafo mt pequeno"

for c in chunks:
    assert _approx_tokens(c.text) <= CHUNK_SIZE, f"Chunk {c.chunk_index} excede CHUNK_SIZE"
print("OK — parágrafo longo dividido corretamente")

Tokens do parágrafo: ~1354
=== Parágrafo longo ===
Total chunks : 3
Token sizes  : min=388 | max=479 | avg=447

[chunk 0] page=1 | ~476 tokens
'Esta é a sentença número 1 do parágrafo muito longo que precisa ser dividido pelo chunker Esta é a sentença número 2 do parágrafo muito longo que precisa ser dividido pelo chunker Esta é a sentença nú'

[chunk 1] page=1 | ~479 tokens
'Esta é a sentença número 22 do parágrafo muito longo que precisa ser dividido pelo chunker Esta é a sentença número 23 do parágrafo muito longo que precisa ser dividido pelo chunker Esta é a sentença '

[chunk 2] page=1 | ~388 tokens
'Esta é a sentença número 43 do parágrafo muito longo que precisa ser dividido pelo chunker Esta é a sentença número 44 do parágrafo muito longo que precisa ser dividido pelo chunker Esta é a sentença '

OK — parágrafo longo dividido corretamente


## 3 — Overlap: verificação de sobreposição entre chunks consecutivos

In [66]:
# Texto grande o suficiente para gerar pelo menos 3 chunks
sentences = [f"Sentença única e identificável número {i} no documento de teste" for i in range(1, 70)]
text = ". ".join(sentences) + "."

pages = [PageContent(page_number=1, text=text, filename="overlap_test.pdf")]
chunks = chunk_pages(pages,chunk_overlap = 80)
print_chunks(chunks)
print(f"Chunk 0 termina em: {repr(chunks[0].text[-40:])}")
print(f"Chunk 1 começa em : {repr(chunks[1].text[:40])}...")
tail_words = chunks[0].text.split()[-10:]
print(tail_words)

for w in tail_words:
    if w in chunks[1].text:
        print('sim')
chunks[0].text.split()[-10:]

[chunk 0] page=1 | ~503 tokens
'Sentença única e identificável número 1 no documento de teste Sentença única e identificável número 2 no documento de teste Sentença única e identificável número 3 no documento de teste Sentença única'

[chunk 1] page=1 | ~505 tokens
'Sentença única e identificável número 29 no documento de teste Sentença única e identificável número 30 no documento de teste Sentença única e identificável número 31 no documento de teste Sentença ún'

[chunk 2] page=1 | ~205 tokens
'Sentença única e identificável número 57 no documento de teste Sentença única e identificável número 58 no documento de teste Sentença única e identificável número 59 no documento de teste Sentença ún'

Chunk 0 termina em: 'ificável número 32 no documento de teste'
Chunk 1 começa em : 'Sentença única e identificável número 29'...
['Sentença', 'única', 'e', 'identificável', 'número', '32', 'no', 'documento', 'de', 'teste']
sim
sim
sim
sim
sim
sim
sim
sim
sim
sim


['Sentença',
 'única',
 'e',
 'identificável',
 'número',
 '32',
 'no',
 'documento',
 'de',
 'teste']

In [ ]:
# Texto grande o suficiente para gerar pelo menos 3 chunks
sentences = [f"Sentença única e identificável número {i} no documento de teste" for i in range(1, 70)]
text = ". ".join(sentences) + "."

pages = [PageContent(page_number=1, text=text, filename="overlap_test.pdf")]
chunks = chunk_pages(pages)

print_chunk_summary(chunks, "Teste de overlap")
print(f"Chunk 0 termina em: ...{repr(chunks[0].text[-120:])}")
print()
print(f"Chunk 1 começa em : {repr(chunks[1].text[:120])}...")
print()
print(f"Chunk 2 começa em : {repr(chunks[2].text[:120])}...")
print()
print_chunks(chunks)


assert len(chunks) >= 2, "Deve gerar pelo menos 2 chunks para testar overlap"
# Verifica que algum conteúdo do final do chunk 0 aparece no início do chunk 1
tail_words = chunks[0].text.split()[-10:]
overlap_found = any(w in chunks[1].text for w in tail_words)
print("Overlap detectado entre chunk 0 e chunk 1:", overlap_found)

=== Teste de overlap ===
Total chunks : 3
Token sizes  : min=79 | max=505 | avg=362

Chunk 0 termina em: ...'nça única e identificável número 31 no documento de teste Sentença única e identificável número 32 no documento de teste'

Chunk 1 começa em : 'Sentença única e identificável número 33 no documento de teste Sentença única e identificável número 34 no documento de '...

Chunk 2 começa em : 'Sentença única e identificável número 65 no documento de teste Sentença única e identificável número 66 no documento de '...

[chunk 0] page=1 | ~503 tokens
'Sentença única e identificável número 1 no documento de teste Sentença única e identificável número 2 no documento de teste Sentença única e identificável número 3 no documento de teste Sentença única'

[chunk 1] page=1 | ~505 tokens
'Sentença única e identificável número 33 no documento de teste Sentença única e identificável número 34 no documento de teste Sentença única e identificável número 35 no documento de teste Sentença ún'

[chu

## 4 — Chunks pequenos: mesclagem abaixo de MIN_CHUNK_TOKENS

In [69]:
# Parágrafos minúsculos (bem abaixo de 50 tokens cada)
tiny_paragraphs = [f"Item {i}." for i in range(1, 20)]
text = "\n\n".join(tiny_paragraphs)

pages = [PageContent(page_number=1, text=text, filename="tiny.pdf")]
chunks = chunk_pages(pages)

print(f'total words {len(text)}')
print_chunk_summary(chunks, "Parágrafos minúsculos")
print_chunks(chunks)

# Os 19 parágrafos minúsculos devem ter sido mesclados em bem menos chunks
assert len(chunks) < len(tiny_paragraphs), "Parágrafos minúsculos devem ser mesclados"
print(f"OK — {len(tiny_paragraphs)} parágrafos minúsculos → {len(chunks)} chunk(s)")

total words 179
=== Parágrafos minúsculos ===
Total chunks : 1
Token sizes  : min=44 | max=44 | avg=44

[chunk 0] page=1 | ~44 tokens
'Item 1.\n\nItem 2.\n\nItem 3.\n\nItem 4.\n\nItem 5.\n\nItem 6.\n\nItem 7.\n\nItem 8.\n\nItem 9.\n\nItem 10.\n\nItem 11.\n\nItem 12.\n\nItem 13.\n\nItem 14.\n\nItem 15.\n\nItem 16.\n\nItem 17.\n\nItem 18.\n\nItem 19.'

OK — 19 parágrafos minúsculos → 1 chunk(s)


## 5 — Múltiplas páginas: chunk_index global e page_number correto

In [70]:
pages = [
    PageContent(page_number=1, text="Conteúdo da página 1. " * 30, filename="multipage.pdf"),
    PageContent(page_number=2, text="Conteúdo da página 2. " * 30, filename="multipage.pdf"),
    PageContent(page_number=3, text="Conteúdo da página 3. " * 30, filename="multipage.pdf"),
]

chunks = chunk_pages(pages)

print_chunk_summary(chunks, "Múltiplas páginas")

# chunk_index deve ser sequencial sem gaps
indices = [c.chunk_index for c in chunks]
assert indices == list(range(len(chunks))), "chunk_index deve ser sequencial"

# Cada chunk deve ter page_number correto
page_nums = sorted(set(c.page_number for c in chunks))
print(f"Páginas representadas nos chunks: {page_nums}")
assert set(page_nums) == {1, 2, 3}, "Todas as 3 páginas devem aparecer nos chunks"

print("OK — chunk_index sequencial e page_number correto")

# Distribuição por página
from collections import Counter
dist = Counter(c.page_number for c in chunks)
for pg, count in sorted(dist.items()):
    print(f"  Página {pg}: {count} chunk(s)")

=== Múltiplas páginas ===
Total chunks : 3
Token sizes  : min=165 | max=165 | avg=165

Páginas representadas nos chunks: [1, 2, 3]
OK — chunk_index sequencial e page_number correto
  Página 1: 1 chunk(s)
  Página 2: 1 chunk(s)
  Página 3: 1 chunk(s)


## 6 — Página vazia: deve ser ignorada silenciosamente

In [71]:
pages = [
    PageContent(page_number=1, text="Conteúdo normal da primeira página.", filename="empty_page.pdf"),
    PageContent(page_number=2, text="   \n\n   ", filename="empty_page.pdf"),   # página em branco
    PageContent(page_number=3, text="Conteúdo normal da terceira página.", filename="empty_page.pdf"),
]

chunks = chunk_pages(pages)

page_nums = {c.page_number for c in chunks}
print(f"Páginas nos chunks: {sorted(page_nums)}")
assert 2 not in page_nums, "Página vazia não deve gerar chunks"
print("OK — página vazia ignorada corretamente")

Páginas nos chunks: [1, 3]
OK — página vazia ignorada corretamente


## 7 — PDFs reais: chunking end-to-end

In [72]:
pdfs = sorted(PDF_DIR.glob("*.pdf"))
print(f"{len(pdfs)} PDFs encontrados:")
for p in pdfs:
    print(" ", p.name)

4 PDFs encontrados:
  LB5001.pdf
  MN414_0224.pdf
  WEG-CESTARI-manual-iom-guia-consulta-rapida-50111652-pt-en-es-web.pdf
  WEG-motores-eletricos-guia-de-especificacao-50032749-brochure-portuguese-web.pdf


In [73]:
for pdf_path in pdfs:
    file_bytes = pdf_path.read_bytes()
    result = extract_pdf(file_bytes, pdf_path.name)

    if not result.has_extractable_text:
        print(f"{pdf_path.name}: sem texto extraível — pulando")
        continue

    chunks = chunk_pages(result.pages)

    token_counts = [_approx_tokens(c.text) for c in chunks]
    oversized = [c for c in chunks if _approx_tokens(c.text) > CHUNK_SIZE]

    print(f"{pdf_path.name}")
    print(f"  Páginas extraídas : {len(result.pages)}")
    print(f"  Chunks gerados    : {len(chunks)}")
    print(f"  Tokens min/max/avg: {min(token_counts)} / {max(token_counts)} / {sum(token_counts)//len(token_counts)}")
    print(f"  Chunks acima do limite: {len(oversized)}")
    print()

LB5001.pdf
  Páginas extraídas : 2
  Chunks gerados    : 4
  Tokens min/max/avg: 133 / 491 / 390
  Chunks acima do limite: 0

MN414_0224.pdf
  Páginas extraídas : 15
  Chunks gerados    : 23
  Tokens min/max/avg: 1 / 504 / 315
  Chunks acima do limite: 0

WEG-CESTARI-manual-iom-guia-consulta-rapida-50111652-pt-en-es-web.pdf
  Páginas extraídas : 81
  Chunks gerados    : 97
  Tokens min/max/avg: 62 / 518 / 334
  Chunks acima do limite: 1

WEG-motores-eletricos-guia-de-especificacao-50032749-brochure-portuguese-web.pdf
  Páginas extraídas : 68
  Chunks gerados    : 144
  Tokens min/max/avg: 1 / 518 / 398
  Chunks acima do limite: 21



## 8 — Inspecionar chunks do primeiro PDF

In [74]:
pdf_path = pdfs[0]
file_bytes = pdf_path.read_bytes()
result = extract_pdf(file_bytes, pdf_path.name)
chunks = chunk_pages(result.pages)

print(f"Mostrando primeiros 5 chunks de '{pdf_path.name}'\n")
print_chunks(chunks, n=5)

Mostrando primeiros 5 chunks de 'LB5001.pdf'

[chunk 0] page=1 | ~491 tokens
'AC & DC Motor Installation & Maintenance Instructions Handling The weight of the motor and shipping container will vary. Use correct material handling equipment to avoid injury. Receiving Inspect the '

[chunk 1] page=1 | ~469 tokens
'each bolt (spring washer or thread lock compound). Motor Enclosure ODP, Open drip proof motors are intended for use in clean, dry locations with adequate supply of cooling air. These motors should not'

[chunk 2] page=2 | ~468 tokens
'P.O. Box 2400 Fort Smith, AR 72902-2400 USA (479) 646-4711 9/14 LB5001 Lubrication This is a ball bearing motor. The bearings have been lubricated at the factory. Motors that do not have regrease capa'

[chunk 3] page=2 | ~133 tokens
'teaspoon Up to 210 incl. (132) 0.30 (8.4) 0.6 2.0 Over 210 to 280 incl. (180) 0.61 (17.4) 1.2 3.9 Over 280 to 360 incl. (225) 0.81 (23.1) 1.5 5.2 Over 360 to 5000 incl.(300) 2.12(60.0) 4.1 13.4 Mainte'



## 9 — Parâmetros customizados: chunk_size menor

In [75]:
pdf_path = pdfs[0]
file_bytes = pdf_path.read_bytes()
result = extract_pdf(file_bytes, pdf_path.name)

chunks_default = chunk_pages(result.pages)
chunks_small   = chunk_pages(result.pages, chunk_size=128, chunk_overlap=20)

print(f"chunk_size=512 → {len(chunks_default)} chunks")
print(f"chunk_size=128 → {len(chunks_small)} chunks")
assert len(chunks_small) > len(chunks_default), "chunk_size menor deve gerar mais chunks"
print("OK — chunk_size menor gera mais chunks")

chunk_size=512 → 4 chunks
chunk_size=128 → 16 chunks
OK — chunk_size menor gera mais chunks
